# Results Summary

This notebook summarizes the results of the recommendation system experiments.

Models compared:

1. Matrix Factorization (MF)
2. Matrix Factorization + Content Features (MF + CBF)
3. Two-Tower Retrieval + MF Reranking

The goal is to compare ranking quality across models using the saved experiment outputs.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
import json
import pickle
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.config import METRICS_DIR, FIGURES_DIR

## Load saved summaries

In [ ]:
with open(METRICS_DIR / "mf_summary.json", "r") as f:
    mf_summary = json.load(f)

with open(METRICS_DIR / "mf_cbf_summary.json", "r") as f:
    mf_cbf_summary = json.load(f)

with open(METRICS_DIR / "two_tower_summary.json", "r") as f:
    two_tower_summary = json.load(f)

mf_summary, mf_cbf_summary, two_tower_summary

In [ ]:
def summary_to_rows(model_name, summary):
    rows = []
    for metric, values in summary.items():
        rows.append(
            {
                "model": model_name,
                "metric": metric,
                "mean": values["mean"],
                "std": values["std"],
            }
        )
    return rows

rows = []
rows.extend(summary_to_rows("MF", mf_summary))
rows.extend(summary_to_rows("MF+CBF", mf_cbf_summary))
rows.extend(summary_to_rows("Two-Tower", two_tower_summary))

results_df = pd.DataFrame(rows)
results_df = results_df.sort_values(["metric", "model"]).reset_index(drop=True)
results_df

## Mean metric table

In [ ]:
pivot_df = results_df.pivot(index="metric", columns="model", values="mean").round(4)
pivot_df

## Standard deviation table

In [ ]:
std_df = results_df.pivot(index="metric", columns="model", values="std").round(4)
std_df

## Plot comparison by metric

In [ ]:
for metric in results_df["metric"].unique():
    metric_df = results_df[results_df["metric"] == metric]

    plt.figure(figsize=(7, 4))
    plt.bar(metric_df["model"], metric_df["mean"], yerr=metric_df["std"], capsize=5)
    plt.title(f"Model Comparison - {metric}")
    plt.xlabel("Model")
    plt.ylabel(metric)
    plt.tight_layout()
    plt.show()

## Load full run results

In [ ]:
with open(METRICS_DIR / "mf_results.pkl", "rb") as f:
    mf_results = pickle.load(f)

with open(METRICS_DIR / "mf_cbf_results.pkl", "rb") as f:
    mf_cbf_results = pickle.load(f)

with open(METRICS_DIR / "two_tower_results.pkl", "rb") as f:
    two_tower_results = pickle.load(f)

print("MF runs:", len(mf_results))
print("MF+CBF runs:", len(mf_cbf_results))
print("Two-Tower runs:", len(two_tower_results))

## Example training curves
This section shows how metrics evolved during training for one sample run from each model.

In [ ]:
sample_mf = mf_results[0]
sample_mf_cbf = mf_cbf_results[0]
sample_tt = two_tower_results[0]

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(sample_mf["history"]["train_loss"], label="MF Train Loss")
plt.plot(sample_mf["history"]["val_mse"], label="MF Val MSE")
plt.title("MF Training History")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(sample_mf_cbf["history"]["train_loss"], label="MF+CBF Train Loss")
plt.plot(sample_mf_cbf["history"]["val_mse"], label="MF+CBF Val MSE")
plt.title("MF+CBF Training History")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(sample_tt["retrieval_history"]["train_loss"], label="Two-Tower Train Loss")
plt.plot(sample_tt["retrieval_history"]["val_loss"], label="Two-Tower Val Loss")
plt.title("Two-Tower Retrieval History")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

## Final observations

Main takeaways from the experiments:

- MF provides a strong collaborative filtering baseline
- MF + CBF adds side information, but does not outperform MF in this setup
- the Two-Tower pipeline gives the best overall ranking quality
- retrieval + reranking is a stronger system design than using a single recommendation model alone

This matches the overall goal of the project: moving from simple recommendation models toward a more realistic multi-stage recommendation pipeline.